# Mixtral of Experts (Mixtral 8x7B)

http://arxiv.org/abs/2401.04088

In [2]:
import math
from dataclasses import dataclass

import torch
from torch import nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
@dataclass
class ModelConfig:

    dim=4096
    n_layers=4              # 32
    head_dim=6128           # dim // n_heads
    hidden_dim=14336
    n_heads=32
    n_kv_heads=8
    context_len = 32_768
    vocab_size=32_000
    n_experts=8
    n_experts_per_tok=2
    attn_dropout=0.1

In [5]:
config = ModelConfig()

In [6]:
bs = 2
seq_len = 16

In [7]:
inputs = torch.randn((bs, seq_len, config.dim))

In [8]:
def precompute_freqs_cis(
        seq_len: int,
        head_dim: int,
        base: int = 10_000
) -> torch.Tensor:
    freqs = 1.0 / (base ** (torch.arange(0, head_dim, 2)[: (head_dim // 2)].float() / head_dim))    # (d // 2)
    t = torch.arange(seq_len, device=freqs.device)                                                 # [0, 1, 2, ..., seq_len-1], (seql_len)
    freqs = torch.outer(t, freqs)                                                                  # (seq_len, d // 2)
    # x' = x * cos(theta) + x * sin(theta) * i
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)                                         # (seq_len, d // 2)
    cache = torch.stack([freqs_cis.real, freqs_cis.imag], dim=-1)                                  # (seq_len, d // 2, 2)
    return cache.to(dtype=torch.bfloat16)

In [9]:
freqs_cis = precompute_freqs_cis(seq_len=seq_len, head_dim=config.head_dim)

In [10]:
def apply_rotary_emb(
    x: torch.Tensor,                                                                          # (bs, block_size, n_heads, head_dim)
    freqs_cis: torch.Tensor                                                                   # (block_size, head_dim // 2, 2)
) -> torch.Tensor:
    xshaped = x.float().reshape(*x.shape[:-1], -1, 2)                                          # (bs, block_size, n_heads, head_dim // 2, 2)
    freqs_cis = freqs_cis.view(1, xshaped.size(1), 1, xshaped.size(3), 2)                     # (1, block_size, 1, head_dim // 2, 2)
    x_out = torch.stack(
        [
            # első komponens rotáció: x1 * cos(theta) - x2 * sin(theta)
            xshaped[..., 0] * freqs_cis[..., 0] - xshaped[..., 1] * freqs_cis[..., 1],
            # második komponens rotáció: x2 * cos(theta) + x1 * sin(theta)
            xshaped[..., 1] * freqs_cis[..., 0] + xshaped[..., 0] * freqs_cis[..., 1],
        ],
        -1,
    )
    x_out = x_out.flatten(3)
    return x_out.type_as(x)

In [11]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, dim: int, n_heads: int, head_dim: int, n_kv_heads: int, dropout: float) -> None:
        super().__init__()

        self.n_heads = n_heads
        self.head_dim = head_dim                        # dim // n_heads
        self.n_kv_heads = n_kv_heads                    # n groups

        self.repeats = self.n_heads // self.n_kv_heads  # group size

        self.wq = nn.Linear(dim, n_heads * head_dim, bias=False)
        self.wk = nn.Linear(dim, n_kv_heads * head_dim, bias=False)
        self.wv = nn.Linear(dim, n_kv_heads * head_dim, bias=False)
        self.wo = nn.Linear(n_heads * head_dim, dim, bias=False)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, freqs_cis: torch.Tensor, cache = None, mask = None) -> torch.Tensor:

        assert mask is None or cache is None
        bs, seq_len, _ = x.shape

        q, k, v = self.wq(x), self.wk(x), self.wv(x)

        q = q.view(bs, seq_len, self.n_heads, self.head_dim)
        k = k.view(bs, seq_len, self.n_kv_heads, self.head_dim)
        v = v.view(bs, seq_len, self.n_kv_heads, self.head_dim)

        q = apply_rotary_emb(q, freqs_cis=freqs_cis)
        k = apply_rotary_emb(k, freqs_cis=freqs_cis)

        q, k, v = map(lambda x: x.transpose(1, 2), (q, k, v))

        if cache is not None:
            raise NotImplementedError # TODO

        k = k.repeat_interleave(self.repeats, dim=1)
        v = v.repeat_interleave(self.repeats, dim=1)

        attn_weights = torch.matmul(q, k.transpose(2, 3)) / math.sqrt(self.head_dim)

        if mask is not None:
            raise NotImplementedError # TODO

        attn_weights = F.softmax(attn_weights, dim=-1)
        attn_weights = self.dropout(attn_weights)
        output = torch.matmul(attn_weights, v).transpose(1, 2).contiguous()
        output = output.view(bs, seq_len, self.n_heads * self.head_dim)
        return self.wo(output)

In [12]:
# gqa = GroupedQueryAttention(
#     config.dim, config.n_heads, config.head_dim, n_kv_heads=config.n_kv_heads, dropout=config.attn_dropout
# )

In [13]:
# gqa(inputs, freqs_cis).shape

## Kapuzó előrecsatolt réteg (gated feed-forward layer)

A hagyományos Transformer MLP-blokk egy kapuzott feed-forward réteggel (Gated Feed-Forward / GLU-szerű MLP) van helyettesítve. Pontosabban annak is egy variánsával (SwiGLU).

A bemeneti rejtett reprezentáció két külön lineáris vetítésen megy keresztül: az egyik ág ($\mathbf{W}_1$) a köztes dimenzióra (`hidden_dim`) vetít és aktivációs függvényen (SiLU) halad át, míg a másik ág ($\mathbf{W}_g$) kapuként múködik aktiváció nélkül.

A két ág kimenete elemenkénti szorzással (Hadamard-szorzat) kombinálódik, majd egy kimeneti lineáris réteg ($\mathbf{W}_2$) visszavetíti az eredményt az eredeti rejtett dimenzióra. Bővebben lásd [itt](https://arxiv.org/abs/2002.05202) és a publikáció kódját [itt](https://github.com/mistralai/mistral-inference/tree/main).

In [14]:
class GatedFeedForward(nn.Module):
    def __init__(self, dim: int, hidden_dim: int) -> None:
        super().__init__()

        self.w1 = nn.Linear(dim, hidden_dim, bias=False)
        self.wg = nn.Linear(dim, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, dim, bias=False)
        self.act_fn = nn.SiLU()

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        # (bs, seq_len, hidden_dim) = (bs, seq_len, dim) @ (dim, hidden_dim)
        hidden_states = self.act_fn(self.w1(inputs)) # up
        # (bs, seq_len, hidden_dim) = (bs, seq_len, hidden_dim) * (bs, seq_len, hidden_dim)
        hidden_states = hidden_states * self.wg(inputs) # gate
        # (bs, seq_len, dim) = (bs, seq_len, hidden_dim) @ (hidden_dim, dim)
        hidden_states = self.w2(hidden_states) # down
        return hidden_states

In [15]:
class RMSNorm(torch.nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6) -> None:
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.variance_epsilon = eps

    def forward(self, hidden_states):
        input_dtype = hidden_states.dtype
        hidden_states = hidden_states.to(torch.float32)
        variance = hidden_states.pow(2).mean(-1, keepdim=True)
        hidden_states = hidden_states * torch.rsqrt(variance + self.variance_epsilon)
        return self.weight * hidden_states.to(input_dtype)


## Mixture of Expert (MoE) réteg

Az eredeti Transzformer architektúra dekóder részéből a teljesen összekapcsolt előrecsatolt réteg le lesz cserélve egy ún. Mixture-of-Expert rétegre. A réteg alapvetően egy két új résszel és - ennek megfelelően - két feladattal bővül.

Az első rész, hogy egy előrecsatolt réteg helyett összesen $n$ darab réteg lesz és mindegyiket szakértőknek (expert) nevezzük. A második komponenst útválasztási vagy irányítási (routing) mechanizmusnak (vagy hálózatnak) hívják és feladata annak eldöntése, hogy melyik token melyik szakértőhöz kerüljön feldolgozásra. Egy token összesen $m$ szakértőhez kerülhet ($m \ll n$). Tehát mechanizmus nem szekvenciákat "küld" a szakértő modellekhez, hanem tokeneket, vagyis a feldolgozás token szinten történik. Ez az első funkciója.

Az irányítási hálózat másik szerepe, hogy a szakértők által adott kimeneteket súlyozza a saját (súly)mátrixában lévő megfelelő indexű súllyal (lásd lentebb).

Formálisan:

Legyen $\mathbf{X} \in \mathbb{R}^{B \times T \times d}$ a bemeneti mátrix, ahol $b \in\{1, \ldots, B\}$ a batch indexe, $t \in\{1, \ldots, T\}$ a szekvencia indexe (vagyis az adott token) és $x_{b, t} \in \mathbb{R}^d$ a $t$-edik token rejtett reprezentációja (vektora) a $b$-edik példában.

Legyen az irányítási réteg egy $G:  \mathbb{R}^d \rightarrow  \mathbb{R}^n$ leképezés, ahol $n$ a szakértők száma, amit minden tokenenre alkalmazunk az alábbi módon:

$$
G(x) := \operatorname{Softmax}\left(\operatorname{TopK}\left(x \cdot \mathbf{W}_g\right)\right).
$$

Mivel ez is egy réteg, ezért rendelkezik a $\mathbf{W}_g \in \mathbb{R}^{d \times n}$ súlymátrixszal.

A képletben három fontos lépés történik, amiket az inferenciális ugrások elkerülése érdekében leírok.

Az első a $x \cdot \mathbf{W}_g$ művelet, aminek eredménye - az adott tokenhez tartozó - $n$-dimenziós vektor. Ez lesz az ún. irányítási érték (routing score), amely később a szakértők kimeneteinek súlyozásához szolgál.

A második lépés a $\operatorname{TopK}$ művelet alkalmazása az első lépés eredményére. Az operátor egy ritka (sparse) (súly)vektort hoz létre: csak $m$ komponense nem nulla, és ezek a legnagyobb értékkel rendelkező elemekhez tartoznak.

*Megjegyzés: A PyTorch itt két változót ad vissza, egyet az értékeknek és egyet a kiválasztott szakértők indexeknek*.

A harmadik lépésben a softmax aktivációs függvény segítségével az $m$ dimenziós vektort valószínűségi eloszlássá alakítjuk.

Ha az $n$ darab szakértőt $\left\{E_1, \ldots, E_n\right\}$ jelöli, ahol minden

$$
E_i: \mathbb{R}^d \rightarrow \mathbb{R}^d,
$$

akkor a MoE réteg kimenetét a következő súlyozott összeg adja:

$$
\operatorname{MoE}(x):=\sum_{i=1}^n G_i(x) E_i(x) .
$$

Itt a $G_i(x)$ (az $i$-edik szakértőhoz tartozó) skalár súly, és a $G_i(x) E_i(x)$ szorzás skalár-vektor szorzást jelent (mivel $E_i(x) \in \mathbb{R}^d$) és $i$ a szakértő indexe.

Részletesen lásd [itt](http://arxiv.org/abs/2209.01667).

In [16]:
class MoELayer(nn.Module):
    def __init__(self, dim: int, hidden_dim: int, n_experts: int, n_experts_per_tok: int = 2) -> None:
        super().__init__()

        self.n_experts = n_experts
        self.n_experts_per_tok = n_experts_per_tok

        # gating, router, routing function (dim -> num_experts)
        self.gate = nn.Linear(dim, n_experts, bias=False)
        self.experts = nn.ModuleList([
            GatedFeedForward(dim=dim, hidden_dim=hidden_dim) for _ in range(n_experts)
        ])

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:

        batch_size, seq_len, dim = inputs.shape

        # gated routing is token-wise, not sequence-wise, i.e.
        # every token in the sequence is routed independently to n experts
        inputs = inputs.view(-1, dim)  # (batch_size * seq_len, dim)

        # output of the gating network
        router_logits = self.gate(inputs) # (batch_size * seq_len, n_experts)

        # select top n experts for every token along with their weights
        # (batch_size * seq_len, num_experts_per_tok), (batch_size * seq_len, num_experts_per_tok)
        routing_weights, selected_experts = torch.topk(router_logits, self.n_experts_per_tok)

        # (batch_size * seq_len, num_experts_per_tok)
        routing_weights = F.softmax(routing_weights, dim=1, dtype=torch.float).to(inputs.dtype)

        results = torch.zeros_like(inputs, dtype=inputs.dtype, device=inputs.device) # (batch_size * seq_len, dim)

        for i, expert in enumerate(self.experts):
            # select the tokens assigned to expert i (avoid computing irrelevant experts)
            token_idx, nth_expert = torch.where(selected_experts == i)

            # selected routing weights (scalars) for the chosen tokens
            r = routing_weights[token_idx, nth_expert, None] # (n, 1)
            e = expert(inputs[token_idx]) # (n, dim) <- expert(n, dim)
            # element-wise multiplication (with 1 broadcasted along dim)
            # (n, dim) = (n, 1) * (n, dim)
            results[token_idx] += r * e

        results = results.reshape(batch_size, seq_len, dim)
        return results

In [17]:
# x = torch.randn((2, 4, 4)) # (batch_size, seq_len, dim)
# moe_layer = MoELayer(dim=4, hidden_dim=8, n_experts=5, n_experts_per_tok=2)

In [18]:
# moe_layer(x).shape

In [19]:
class DecoderLayer(nn.Module):
    def __init__(
            self,
            dim: int,
            hidden_dim: int,
            n_heads: int,
            n_kv_heads: int,
            head_dim: int,
            attn_dropout: float,
            n_experts: int,
            n_experts_per_tok: int,
        ) -> None:
        super().__init__()

        self.n_heads = n_heads
        self.dim = dim
        self.attention = GroupedQueryAttention(dim, n_heads, head_dim, n_kv_heads, attn_dropout)
        self.attention_norm = RMSNorm(dim)
        self.ffn_norm = RMSNorm(dim)

        self.feed_forward = MoELayer(dim, hidden_dim, n_experts, n_experts_per_tok)

    def forward(self, x: torch.Tensor, freqs_cis: torch.Tensor) -> torch.Tensor:

        residual = x
        hidden_states = self.attention_norm(x)
        hidden_states = self.attention(hidden_states, freqs_cis)
        hidden_states = hidden_states + residual

        residual = hidden_states
        hidden_states = self.feed_forward(self.ffn_norm(hidden_states))
        out = hidden_states + residual
        return out


In [20]:
# layer = DecoderLayer(
#     dim=config.dim, hidden_dim=config.hidden_dim, n_heads=config.n_heads,
#     head_dim=config.head_dim, n_kv_heads=config.n_kv_heads, attn_dropout=config.attn_dropout,
#     n_experts=config.n_experts, n_experts_per_tok=config.n_experts_per_tok,
# )

In [21]:
# layer(inputs, freqs_cis).shape

In [22]:
class MixtureOfExperts(nn.Module):
    def __init__(
            self,
            vocab_size: int,
            n_layers: int,
            dim: int,
            hidden_dim: int,
            n_heads: int,
            n_kv_heads: int,
            head_dim: int,
            attn_dropout: float,
            n_experts: int,
            n_experts_per_tok: int,
        ) -> None:
        super().__init__()

        self.head_dim = head_dim

        self.tok_embeddings = nn.Embedding(vocab_size, dim)

        self.layers = nn.ModuleList([
            DecoderLayer(
                dim=dim,
                hidden_dim=hidden_dim,
                n_heads=n_heads,
                n_kv_heads=n_kv_heads,
                head_dim=head_dim,
                attn_dropout=attn_dropout,
                n_experts=n_experts,
                n_experts_per_tok=n_experts_per_tok,
            ) for _ in range(n_layers)
        ])

        self.final_norm = RMSNorm(dim)
        self.output = nn.Linear(dim, vocab_size, bias=False)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:

        freqs_cis = precompute_freqs_cis(input_ids.shape[1], head_dim=self.head_dim)
        freqs_cis = freqs_cis.to(input_ids.device)

        hidden_states = self.tok_embeddings(input_ids)
        for layer in self.layers:
            hidden_states = layer(hidden_states, freqs_cis)
        hidden_states = self.final_norm(hidden_states)
        outs = self.output(hidden_states)
        return outs

    @property
    def device(self) -> torch.device:
        return next(self.parameters()).device

    @classmethod
    def from_pretrained(cls, model_type):

        from transformers import AutoModelForCausalLM

        config = ModelConfig()
        model = MixtureOfExperts(**config)
        sd = model.state_dict()

        model_hf = AutoModelForCausalLM.from_pretrained(model_type)
        sd_hf = model_hf.state_dict()

        raise NotImplementedError # TODO

In [23]:
model = MixtureOfExperts(
    vocab_size=config.vocab_size,
    n_layers=config.n_layers,
    dim=config.dim,
    hidden_dim=config.hidden_dim,
    n_heads=config.n_heads,
    n_kv_heads=config.n_kv_heads,
    head_dim=config.head_dim,
    attn_dropout=config.attn_dropout,
    n_experts=config.n_experts,
    n_experts_per_tok=config.n_experts_per_tok,
)
model = model.to(device)

In [24]:
# sd = model.state_dict()

In [25]:
inputs = torch.randint(0,10_000, (bs, seq_len), device=model.device, dtype=torch.long)

In [26]:
model(inputs).shape

torch.Size([2, 16, 32000])

In [30]:
@torch.inference_mode()
def generate(tokens, model, max_tokens, temperature=1.0, do_sample=False, top_k=None, max_seq_len = 1024):

    tokens_batch = tokens.unsqueeze(0).to(model.device)

    for _ in range(max_tokens):

        # cut context if too long
        tokens_batch = tokens_batch if tokens_batch.size(1) <= max_seq_len else tokens_batch[:, -max_seq_len:]

        logits = model(tokens_batch)

        logits = logits[:, -1, :] / temperature

        if top_k is not None:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = -float('Inf')

        probs = F.softmax(logits, dim=-1)

        # either sample from the distribution or take the most likely element
        if do_sample:
            token_next = torch.multinomial(probs, num_samples=1)
        else:
            _, token_next = torch.topk(probs, k=1, dim=-1)

        tokens_batch = torch.cat((tokens_batch, token_next), dim=1)
    return tokens_batch

In [31]:
tokens = torch.tensor([1,2,3])

In [34]:
generate(tokens, model, max_tokens=10, temperature=0.7) # dummy generation

tensor([[    1,     2,     3, 31151, 12221,  5790, 17763, 31564,  9837,  7313,
         31379, 11811,  9057]], device='cuda:0')